In [2]:
import pandas as pd
import datetime as dt

In [3]:
df=pd.read_csv('C:\\Users\\jahnavi\\OneDrive\\Desktop\\New folder (2)\\sample superstore dataset.csv',encoding='latin-1')

In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=False)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed', dayfirst=False)


In [7]:
snapshot_date=df['Order Date'].max()+dt.timedelta(days=1)
print("snapshot date:",snapshot_date)

snapshot date: 2017-12-31 00:00:00


In [25]:
rfm=df.groupby('Customer ID').agg(
    recency=('Order Date',lambda x:(snapshot_date-x.max()).days),
    frequency=('Order ID','nunique'),
    monetary=('Sales','sum')
).reset_index()

In [26]:
print("\nrfm table sample:")
print(rfm.head(10))
print("\nrfm stats:")
print(rfm.describe())


rfm table sample:
  Customer ID  recency  frequency   monetary
0    AA-10315      185          5   5563.560
1    AA-10375       20          9   1056.390
2    AA-10480      260          4   1790.512
3    AA-10645       56          6   5086.935
4    AB-10015      416          3    886.156
5    AB-10060       55          8   7755.620
6    AB-10105       42         10  14473.571
7    AB-10150       42          5    966.710
8    AB-10165       26          8   1113.838
9    AB-10255      167          9    914.532

rfm stats:
           recency   frequency      monetary
count   793.000000  793.000000    793.000000
mean    147.802018    6.316520   2896.848500
std     186.211051    2.550885   2628.670117
min       1.000000    1.000000      4.833000
25%      31.000000    5.000000   1146.050000
50%      76.000000    6.000000   2256.394000
75%     184.000000    8.000000   3785.276000
max    1166.000000   17.000000  25043.050000


In [27]:
rfm['R_Score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score']=pd.qcut(rfm['frequency'].rank(method='first'),q=5,labels=[1,2,3,4,5]).astype(int)
rfm['M_Score']=pd.qcut(rfm['monetary'],q=5,labels=[1,2,3,4,5]).astype(int)

In [28]:
rfm['RFM_Score']=rfm['R_Score']+rfm['F_Score']+rfm['M_Score']

In [29]:
def segment_customer(score):
    if score>=12:
        return 'Champions'
    elif score>=9:
        return 'Loyal Customers'
    elif score>=6:
        return 'At-Risk customers'
    else:
        return 'Lost Customers'

In [30]:
rfm['Segment']=rfm['RFM_Score'].apply(segment_customer)

In [31]:
print("\nRFM Scores sampe:")
print(rfm.head(10))
print("\nSegment Distribution:")
print(rfm['Segment'].value_counts())
print("\n average RFM by segment:")
print(rfm.groupby('Segment')[['recency','frequency','monetary']].mean().round(1))



RFM Scores sampe:
  Customer ID  recency  frequency   monetary  R_Score  F_Score  M_Score  \
0    AA-10315      185          5   5563.560        2        2        5   
1    AA-10375       20          9   1056.390        5        5        2   
2    AA-10480      260          4   1790.512        1        1        3   
3    AA-10645       56          6   5086.935        3        3        5   
4    AB-10015      416          3    886.156        1        1        1   
5    AB-10060       55          8   7755.620        3        4        5   
6    AB-10105       42         10  14473.571        4        5        5   
7    AB-10150       42          5    966.710        4        2        2   
8    AB-10165       26          8   1113.838        5        4        2   
9    AB-10255      167          9    914.532        2        5        1   

   RFM_Score            Segment  
0          9    Loyal Customers  
1         12          Champions  
2          5     Lost Customers  
3         11    Loy

In [ ]:
rfm.to_csv('rfm_segments.csv', index=False)
print("\nSaved to rfm_segments.csv!")